# Neurosurgical Site KOL Ranking – Full Pipeline

This notebook runs the complete pipeline end-to-end on a small example dataset.
It is designed to run in **Google Colab** but works equally well in a local Jupyter environment.

## Steps
1. Install dependencies
2. Clone the repository (or upload files)
3. Retrieve PubMed papers
4. Enrich with OpenAlex metadata
5. Classify papers by surgical relevance
6. Build author network & compute metrics
7. Match author affiliations to canonical sites
8. Compute site-level KOL scores
9. List top-30-centile KOL candidates (site-affiliated)
10. Display and download results

---
**API keys** – PubMed E-utilities work without a key at low volume.  
OpenAlex is fully open – no key required.  
The optional LLM step requires an `OPENAI_API_KEY` (set via Colab secrets or environment).


In [1]:
# ── 1. Install dependencies ──────────────────────────────────────────────────
# Locally: activate the project venv and run once:
#   python -m venv .venv && .venv\Scripts\pip install ipykernel -r requirements.txt
# Then select the .venv kernel in VS Code (Select Kernel → Python Environments).
#
# In Colab: uncomment and run the block below.
# import subprocess, sys
# def install(pkg):
#     subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
# for pkg in ['requests', 'pandas', 'networkx', 'scipy', 'rapidfuzz', 'biopython', 'tqdm']:
#     install(pkg)

print('Dependencies ready.')


Dependencies ready.


In [17]:
# # ── 2. Clone the repository (Colab only) ─────────────────────────────────────
# import os

# REPO_URL = 'https://github.com/roschkoenig/neurosurgical-site-ranking.git'
# REPO_DIR = '/content/neurosurgical-site-ranking'

# if not os.path.exists(REPO_DIR):
#     subprocess.check_call(['git', 'clone', REPO_URL, REPO_DIR])

# os.chdir(REPO_DIR)
# sys.path.insert(0, REPO_DIR)
# print('Working directory:', os.getcwd())

In [2]:
# ── Configuration ────────────────────────────────────────────────────────────
import logging
import os
import sys
from pathlib import Path

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

# Resolve project root both locally and in Colab (folder containing src/ and data/)
cwd = Path.cwd()
candidates = [cwd, cwd.parent]
PROJECT_ROOT = next((p for p in candidates if (p / 'src').is_dir() and (p / 'data').is_dir()), cwd)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Keep working directory at project root so relative output paths are consistent
os.chdir(PROJECT_ROOT)

# Optional: set your NCBI API key for higher throughput
# os.environ['NCBI_API_KEY'] = 'YOUR_KEY_HERE'

# Optional: set your email for OpenAlex polite pool
# os.environ['OPENALEX_EMAIL'] = 'you@example.com'

# LLM matching configuration
USE_LLM = True
LLM_PROVIDER = 'ollama'        # 'openai' or 'ollama'
LLM_MODEL = 'llama3.1:8b'      # OpenAI model or local Ollama model
OLLAMA_URL = 'http://localhost:11434'

# Optional for OpenAI provider only:
# os.environ['OPENAI_API_KEY'] = 'YOUR_KEY_HERE'

MAX_PUBMED_RESULTS = 1500   # small demo; increase to 500+ for real runs

# Targeted enrichment settings
# When True: after the first scoring pass, missing candidate sites are
# automatically searched in PubMed using all their known aliases, and the
# full pipeline is re-run on the combined paper set.
RUN_MISSING_SITE_ENRICHMENT = True
MAX_ENRICHMENT_PMIDS_PER_SITE = 100

ALIASES_CSV = str(PROJECT_ROOT / 'data' / 'site_aliases.csv')
LONGLIST_CSV = str(PROJECT_ROOT / 'data' / 'site_longlist.csv')

print('Configuration done.')
print('Project root:', PROJECT_ROOT)
print('LLM provider:', LLM_PROVIDER)
print('LLM model:', LLM_MODEL)
print('Targeted enrichment enabled:', RUN_MISSING_SITE_ENRICHMENT)


Configuration done.
Project root: c:\Users\Richard Rosch\Documents\Code\neurosurgical-site-ranking
LLM provider: ollama
LLM model: llama3.1:8b
Targeted enrichment enabled: True


In [3]:
# ── 3. Retrieve PubMed papers ────────────────────────────────────────────────
from src.pubmed_search import PubMedSearcher

searcher = PubMedSearcher()
pmids = searcher.search(max_results=MAX_PUBMED_RESULTS)
print(f'Found {len(pmids)} unique PMIDs')

raw_papers = searcher.fetch_details(pmids)
print(f'Fetched details for {len(raw_papers)} papers')

searcher.save_results(raw_papers)
print('Saved raw PubMed results to outputs/pubmed_raw.json')

INFO src.pubmed_search: PubMed search cache hit: 1501 PMIDs
INFO src.pubmed_search: Total unique PMIDs collected: 1500
INFO src.pubmed_search: Fetching PubMed records 1–200 / 1500


Found 1500 unique PMIDs


INFO src.pubmed_search: Fetching PubMed records 201–400 / 1500
INFO src.pubmed_search: Fetching PubMed records 401–600 / 1500
INFO src.pubmed_search: Fetching PubMed records 601–800 / 1500
INFO src.pubmed_search: Fetching PubMed records 801–1000 / 1500
INFO src.pubmed_search: Fetching PubMed records 1001–1200 / 1500
INFO src.pubmed_search: Fetching PubMed records 1201–1400 / 1500
INFO src.pubmed_search: Fetching PubMed records 1401–1500 / 1500


Fetched details for 1500 papers


INFO src.pubmed_search: Saved 1500 PubMed records to C:\Users\Richard Rosch\Documents\Code\neurosurgical-site-ranking\outputs\pubmed_raw.json


Saved raw PubMed results to outputs/pubmed_raw.json


In [4]:
# ── 4. Enrich with OpenAlex metadata ─────────────────────────────────────────
from src.openalex_client import OpenAlexClient

client = OpenAlexClient()
papers = client.enrich_papers(raw_papers)
print(f'Enriched {len(papers)} papers with OpenAlex metadata')

INFO src.openalex_client: Enriching paper 1/1500 (pmid=39180640)
INFO src.openalex_client: Enriching paper 2/1500 (pmid=39983991)
INFO src.openalex_client: Enriching paper 3/1500 (pmid=39440710)
INFO src.openalex_client: Enriching paper 4/1500 (pmid=39868624)
INFO src.openalex_client: Enriching paper 5/1500 (pmid=41444392)
INFO src.openalex_client: Enriching paper 6/1500 (pmid=40684232)
INFO src.openalex_client: Enriching paper 7/1500 (pmid=39128802)
INFO src.openalex_client: Enriching paper 8/1500 (pmid=39527541)
INFO src.openalex_client: Enriching paper 9/1500 (pmid=40007067)
INFO src.openalex_client: Enriching paper 10/1500 (pmid=39327682)
INFO src.openalex_client: Enriching paper 11/1500 (pmid=40973061)
INFO src.openalex_client: Enriching paper 12/1500 (pmid=39111285)
INFO src.openalex_client: Enriching paper 13/1500 (pmid=40211994)
INFO src.openalex_client: Enriching paper 14/1500 (pmid=39341660)
INFO src.openalex_client: Enriching paper 15/1500 (pmid=39179711)
INFO src.openalex_c

Enriched 1500 papers with OpenAlex metadata


In [5]:
# ── 5. Classify papers by surgical relevance ─────────────────────────────────
import pandas as pd
from src.paper_classifier import PaperClassifier

clf = PaperClassifier()
papers = clf.classify_all(papers)

papers_df = pd.DataFrame([
    {
        'pmid': p.get('pmid', ''),
        'title': p.get('title', ''),
        'journal': p.get('journal', ''),
        'year': p.get('year', ''),
        'doi': p.get('doi', ''),
        'cited_by_count': p.get('cited_by_count', 0),
        'label': p.get('label', ''),
        'paper_weight': p.get('paper_weight', 0),
        'openalex_id': p.get('openalex_id', ''),
    }
    for p in papers
])

print(papers_df['label'].value_counts())
papers_df.to_csv('outputs/papers.csv', index=False)
print('Saved outputs/papers.csv')
papers_df.head()

label
surgery_adjacent    943
non_core            328
core_surgical       229
Name: count, dtype: int64
Saved outputs/papers.csv


,pmid,title,journal,year,doi,cited_by_count,label,paper_weight,openalex_id
0,39180640,Metabolic signatures derived from whole-brain ...,Journal of neuro-oncology,2024,10.1007/s11060-024-04812-1,13,non_core,0.0,https://openalex.org/W4401853336
1,39983991,Reproduction of Original Glioblastoma and Brai...,World neurosurgery,2025,10.1016/j.wneu.2025.123808,2,core_surgical,3.0,https://openalex.org/W4407862066
2,39440710,Nature and impact of symptoms at time of initi...,Journal of medical imaging and radiation oncology,2025,10.1111/1754-9485.13796,1,surgery_adjacent,1.0,https://openalex.org/W3209799149
3,39868624,HOTAIR expression as a biomarker in glioblasto...,Biomarkers in medicine,2025,10.1080/17520363.2025.2455925,2,surgery_adjacent,1.0,https://openalex.org/W4406874637
4,41444392,PLX3397 attenuated tumor growth and remodeled ...,Scientific reports,2025,10.1038/s41598-025-32943-6,0,surgery_adjacent,1.0,https://openalex.org/W7117148423


In [6]:
# ── 6. Build author network & compute metrics ─────────────────────────────────
from src.author_network import AuthorNetwork

net = AuthorNetwork()
net.build(papers)
authors_df = net.compute_metrics()
print(f'Author network: {len(authors_df)} authors')
authors_df.head(10)

INFO src.author_network: Built author graph: 10824 nodes, 119140 edges


Author network: 10824 authors


,author_id,display_name,weighted_citation_score,core_surgical_count,surgery_adjacent_count,recency_score,centrifugal_score,degree_centrality,pagerank,affiliations,author_kol_score
0,A5023573690,Bjarne Winther Kristensen,164.00,4,4,10.6018,12.2500,0.005451,0.000406,University of Copenhagen|DK; Copenhagen Univer...,71.31
1,A5021656597,Patrick Y. Wen,79.88,3,9,10.5733,11.8750,0.028827,0.000646,Brigham and Women's Hospital|US; Dana-Farber C...,61.40
2,A5060195279,Jordina Rincón-Torroella,15.42,7,0,8.4273,9.2344,0.005636,0.000367,Johns Hopkins Medicine|US; Johns Hopkins Unive...,54.94
3,A5088235927,Yoshua Esquenazi,22.59,5,1,10.6476,12.0078,0.004897,0.000291,The University of Texas Health Science Center|...,52.38
4,A5017177249,Michael Weller,91.76,3,6,6.7757,7.7539,0.014691,0.000529,University of Zurich|CH; University Hospital o...,51.57
5,A5069558252,Chetan Bettegowda,16.69,6,1,7.8659,8.8125,0.013767,0.000413,Johns Hopkins University|US; Johns Hopkins Med...,51.49
6,A5038354675,Shawn L. Hervey‐Jumper,36.02,4,6,7.8917,9.0651,0.014321,0.000457,"University of California, San Francisco|US; Un...",48.34
7,A5040308869,Peiwen Chen,252.00,0,5,3.4463,4.2500,0.002402,0.000215,Cleveland Clinic Lerner College of Medicine|US...,48.10
8,A5006742893,Stephen Bagley,64.22,3,5,7.0403,7.6411,0.021343,0.000497,University of Pennsylvania|US; Translational T...,47.26
9,A5113360919,Alexandre Roux,2.81,4,0,8.3865,9.1875,0.003696,0.000204,Inserm|FR; Université Paris Cité|FR; Centre Ho...,38.50


In [7]:
# ── 7. Match author affiliations to canonical sites ───────────────────────────
from src.site_matcher import SiteMatcher

matcher = SiteMatcher(
    ALIASES_CSV,
    use_llm=USE_LLM,
    llm_provider=LLM_PROVIDER,
    openai_model=LLM_MODEL,
    ollama_model=LLM_MODEL,
    ollama_url=OLLAMA_URL,
    llm_threshold=88,
 )
authors_df = matcher.match_authors(authors_df)

print('Match method distribution:')
print(authors_df['match_method'].value_counts())

net.save_authors_csv(authors_df)         # outputs/authors.csv
matcher.save_audit()                     # outputs/match_audit.csv
matcher.save_unresolved(authors_df)      # outputs/unresolved_affiliations.csv

authors_df[authors_df['canonical_site'] != ''].head(10)

INFO src.site_matcher: Loaded 123 canonical sites, 564 aliases
WARNING src.site_matcher: Ollama unavailable; LLM step skipped: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError("HTTPConnection(host='localhost', port=11434): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it"))
INFO src.author_network: Saved 10824 author rows to C:\Users\Richard Rosch\Documents\Code\neurosurgical-site-ranking\outputs\authors.csv
INFO src.site_matcher: Saved 15903 audit rows to C:\Users\Richard Rosch\Documents\Code\neurosurgical-site-ranking\outputs\match_audit.csv
INFO src.site_matcher: Saved 5955 unresolved rows to C:\Users\Richard Rosch\Documents\Code\neurosurgical-site-ranking\outputs\unresolved_affiliations.csv


Match method distribution:
match_method
unresolved     5955
exact_alias    2778
fuzzy          2091
Name: count, dtype: int64


,author_id,display_name,weighted_citation_score,core_surgical_count,surgery_adjacent_count,recency_score,centrifugal_score,degree_centrality,pagerank,affiliations,author_kol_score,canonical_site,site_confidence,match_method
0,A5023573690,Bjarne Winther Kristensen,164.00,4,4,10.6018,12.2500,0.005451,0.000406,University of Copenhagen|DK; Copenhagen Univer...,71.31,"University of Colorado Hospital, CO",0.719,fuzzy
1,A5021656597,Patrick Y. Wen,79.88,3,9,10.5733,11.8750,0.028827,0.000646,Brigham and Women's Hospital|US; Dana-Farber C...,61.40,"Brigham and Women's Hospital, Boston",1.000,exact_alias
2,A5060195279,Jordina Rincón-Torroella,15.42,7,0,8.4273,9.2344,0.005636,0.000367,Johns Hopkins Medicine|US; Johns Hopkins Unive...,54.94,"Johns Hopkins University, Baltimore",1.000,exact_alias
3,A5088235927,Yoshua Esquenazi,22.59,5,1,10.6476,12.0078,0.004897,0.000291,The University of Texas Health Science Center|...,52.38,"University of Texas Health Science Center, Hou...",0.950,exact_alias
5,A5069558252,Chetan Bettegowda,16.69,6,1,7.8659,8.8125,0.013767,0.000413,Johns Hopkins University|US; Johns Hopkins Med...,51.49,"Johns Hopkins University, Baltimore",1.000,exact_alias
6,A5038354675,Shawn L. Hervey‐Jumper,36.02,4,6,7.8917,9.0651,0.014321,0.000457,"University of California, San Francisco|US; Un...",48.34,"UCSF, San Francisco",1.000,exact_alias
7,A5040308869,Peiwen Chen,252.00,0,5,3.4463,4.2500,0.002402,0.000215,Cleveland Clinic Lerner College of Medicine|US...,48.10,"Cleveland Clinic, Cleveland",1.000,exact_alias
8,A5006742893,Stephen Bagley,64.22,3,5,7.0403,7.6411,0.021343,0.000497,University of Pennsylvania|US; Translational T...,47.26,"University of Pennsylvania, Philadelphia",1.000,exact_alias
10,A5086562781,Yoshitaka Narita,21.75,2,5,7.2948,8.5078,0.012104,0.000413,National Cancer Centre Japan|JP; National Canc...,36.87,"Institute of Cancer Research, London",0.841,fuzzy
11,A5057551156,Dylan Scott Lykke Harwood,139.25,1,2,3.3618,4.2500,0.002310,0.000141,University of Copenhagen|DK; Copenhagen Univer...,34.13,"University of Colorado Hospital, CO",0.719,fuzzy


In [8]:
# ── 8. Compute site-level composite KOL scores (candidate sites only) ─────────
# SiteScorer:
#   • deduplicates authors by OpenAlex ID then normalised name+site
#   • computes 4 normalised components (neuro strength, network engagement,
#     depth, confidence) and combines them with documented weights
#   • returns ALL candidate sites; unscored sites have NaN scores and a
#     descriptive evidence_status instead of zeros
from src.site_scorer import SiteScorer

paper_label_sets = net.paper_label_sets()   # {author_id: {label: frozenset of paper IDs}}
scorer = SiteScorer(longlist_csv=LONGLIST_CSV, aliases_csv=ALIASES_CSV)
site_scores = scorer.compute(authors_df, paper_label_sets=paper_label_sets)
scorer.save(site_scores)            # outputs/site_scores.csv
scorer.save_possible_duplicates(authors_df)  # outputs/possible_duplicate_authors.csv

scored_count = len(site_scores[site_scores['evidence_status'] == 'scored'])
print(f'Site scores: {scored_count} scored, '
      f'{len(site_scores) - scored_count} unscored '
      f'(of {len(site_scores)} total candidate sites)')
print(f'\nScored sites by country:')
print(site_scores[site_scores['evidence_status'] == 'scored']
      .groupby('country')['canonical_site'].count().to_string())
site_scores[site_scores['evidence_status'] == 'scored'].head(15)


INFO src.site_scorer: Loaded 37 candidate sites from c:\Users\Richard Rosch\Documents\Code\neurosurgical-site-ranking\data\site_longlist.csv
INFO src.site_scorer: Saved 37 site rows to C:\Users\Richard Rosch\Documents\Code\neurosurgical-site-ranking\outputs\site_scores.csv
INFO src.site_scorer: Saved 55 possible duplicate rows to C:\Users\Richard Rosch\Documents\Code\neurosurgical-site-ranking\outputs\possible_duplicate_authors.csv


Site scores: 33 scored, 4 unscored (of 37 total candidate sites)

Scored sites by country:
country
AUS     3
UK      7
USA    23


,canonical_site,country,region,composite_kol_score,site_neurosurgical_kol_strength,site_network_engagement,site_depth,site_confidence,n_kol_candidates,n_core_surgical_papers,n_surgery_adjacent_papers,top_kol_names,evidence_status
0,"Johns Hopkins University, Baltimore",USA,Maryland,95.22,1.0000,1.0000,0.7611,1.0000,88,12,20,Jordina Rincón-Torroella; Chetan Bettegowda; J...,scored
1,"UCSF, San Francisco",USA,California,66.23,0.4761,0.7478,0.8054,1.0000,110,7,30,Shawn L. Hervey‐Jumper; Mitchel S. Berger; Jas...,scored
2,"King's College Hospital, London",UK,Greater London,60.48,0.6685,0.4290,0.7346,0.4985,77,6,14,Honglin Zhu; Antonios N. Pouliopoulos; Yu Wang...,scored
3,"University of Pittsburgh Medical Center, Pitts...",USA,Pennsylvania,59.85,0.4280,0.6728,0.6886,1.0000,61,8,16,Pascal O. Zinn; Megan Mantica; Neslihan Nisa G...,scored
4,"Stanford, Stanford",USA,California,57.18,0.4698,0.5319,0.6373,1.0000,47,8,13,Michael Lim; Richard Drexler; Seunggu J. Han; ...,scored
5,"Cleveland Clinic, Cleveland",USA,Ohio,56.68,0.5083,0.4739,0.6105,0.9749,41,2,20,Peiwen Chen; Yang Liu; Justin D. Lathia; Zhou ...,scored
6,"Mayo Clinic, Jacksonville",USA,Florida,55.82,0.4580,0.5054,0.6287,1.0000,45,4,14,Daniel M. Trifiletti; Kaisorn L. Chaichana; Da...,scored
7,"Northwestern University, Chicago",USA,Illinois,54.27,0.3372,0.5877,0.9923,0.4556,281,6,72,Dieter Henrik Heiland; Amy B. Heimberger; Jawa...,scored
8,"Duke University, Durham",USA,North Carolina,54.02,0.4834,0.5229,0.9575,0.0043,236,12,71,Yuxin Li; Jianpeng Liu; Julia Furtner; Chiman ...,scored
9,"Emory University, Atlanta",USA,Georgia,51.07,0.4165,0.4929,1.0000,0.0000,292,13,58,Ryuta Saito; Jong Hee Chang; Kazuya Motomura; ...,scored


In [9]:
# ── 8a. Missing-site audit + integrated targeted enrichment ──────────────────
# Identifies candidate sites with no matched authors.
# When RUN_MISSING_SITE_ENRICHMENT=True, automatically:
#   1. Retrieves all aliases for missing sites from site_aliases.csv
#   2. Runs targeted PubMed searches using those aliases (not just canonical names)
#   3. Fetches new records and deduplicates against existing raw_papers
#   4. Re-runs enrichment → classification → network → matching → scoring
# No manual cell re-execution needed.

missing_sites_df = scorer.missing_candidates(site_scores)
scorer.save_missing_candidates(site_scores)   # outputs/missing_candidate_sites.csv

print(f'{len(missing_sites_df)} candidate sites not yet scored:')
if not missing_sites_df.empty:
    print(missing_sites_df[['canonical_site', 'country', 'evidence_status']].to_string(index=False))

# Flag for smoke test: did targeted enrichment actually run?
_enrichment_ran = False

if RUN_MISSING_SITE_ENRICHMENT and not missing_sites_df.empty:
    from src.author_network import AuthorNetwork as _ANet

    # ── Step 1: alias-aware targeted search ──────────────────────────────
    missing_site_names = missing_sites_df['canonical_site'].tolist()
    alias_map = matcher.aliases_for_sites(missing_site_names)

    print(f'\nRunning targeted PubMed search for {len(missing_site_names)} missing sites...')
    for site_name, aliases in alias_map.items():
        print(f'  {site_name}: {len(aliases)} aliases')

    enrichment_pmids = searcher.search_by_sites(
        alias_map, max_per_site=MAX_ENRICHMENT_PMIDS_PER_SITE
    )
    existing_pmids = {p.get('pmid', '') for p in raw_papers if p.get('pmid')}
    new_pmids = [p for p in enrichment_pmids if p not in existing_pmids]
    print(f'\nTargeted enrichment: {len(new_pmids)} new unique PMIDs '
          f'(from {len(enrichment_pmids)} total retrieved)')
    _enrichment_ran = True

    if new_pmids:
        # ── Step 2: fetch + append ────────────────────────────────────────
        new_records = searcher.fetch_details(new_pmids)
        raw_papers = raw_papers + new_records
        print(f'raw_papers expanded to {len(raw_papers)} records.')

        # ── Step 3: re-run full pipeline on combined paper set ────────────
        print('\nRe-enriching with OpenAlex (existing papers served from cache)...')
        papers = client.enrich_papers(raw_papers)
        papers = clf.classify_all(papers)
        print(f'Classification: {sum(1 for p in papers if p.get("label") == "core_surgical")} '
              f'core_surgical, {sum(1 for p in papers if p.get("label") == "surgery_adjacent")} '
              f'surgery_adjacent')

        print('Rebuilding author co-authorship network...')
        net = _ANet()
        net.build(papers)
        authors_df = net.compute_metrics()
        print(f'Author network: {len(authors_df)} authors')

        print('Re-matching affiliations (country filter removed; all countries considered)...')
        authors_df = matcher.match_authors(authors_df)
        net.save_authors_csv(authors_df)       # outputs/authors.csv
        matcher.save_audit()                   # outputs/match_audit.csv
        matcher.save_unresolved(authors_df)    # outputs/unresolved_affiliations.csv

        print('Re-scoring candidate sites...')
        paper_label_sets = net.paper_label_sets()
        site_scores = scorer.compute(authors_df, paper_label_sets=paper_label_sets)
        scorer.save(site_scores)
        scorer.save_possible_duplicates(authors_df)

        missing_sites_df = scorer.missing_candidates(site_scores)
        scorer.save_missing_candidates(site_scores)

        scored_after = len(site_scores[site_scores['evidence_status'] == 'scored'])
        print(f'\nAfter enrichment: {scored_after} sites scored, '
              f'{len(missing_sites_df)} still unscored.')
        if not missing_sites_df.empty:
            print('Still unscored:')
            print(missing_sites_df[['canonical_site', 'country', 'evidence_status']]
                  .to_string(index=False))
    else:
        print('No new PMIDs found from targeted enrichment.')
        print('Possible alias gap – check outputs/unresolved_affiliations.csv')
        scorer.save_possible_duplicates(authors_df)

elif not RUN_MISSING_SITE_ENRICHMENT:
    print('\nTargeted enrichment disabled (RUN_MISSING_SITE_ENRICHMENT=False).')
    scorer.save_possible_duplicates(authors_df)
else:
    print('\nAll candidate sites scored – no enrichment needed.')
    scorer.save_possible_duplicates(authors_df)


INFO src.site_scorer: Saved 4 missing candidate rows to C:\Users\Richard Rosch\Documents\Code\neurosurgical-site-ranking\outputs\missing_candidate_sites.csv


4 candidate sites not yet scored:
                  canonical_site country              evidence_status
          Mayo Clinic, Rochester     USA           alias_issue_likely
North Bristol NHS Trust, Bristol      UK           alias_issue_likely
   St Vincent's Hospital, Sydney     AUS           alias_issue_likely
       Austin Health, Heidelberg     AUS no_candidate_authors_matched

Running targeted PubMed search for 4 missing sites...
  Mayo Clinic, Rochester: 4 aliases
  North Bristol NHS Trust, Bristol: 2 aliases
  St Vincent's Hospital, Sydney: 4 aliases
  Austin Health, Heidelberg: 1 aliases


INFO src.pubmed_search: Site enrichment 'Mayo Clinic, Rochester' (4 aliases): 100 PMIDs found
INFO src.pubmed_search: Site enrichment 'North Bristol NHS Trust, Bristol' (2 aliases): 7 PMIDs found
INFO src.pubmed_search: Site enrichment 'St Vincent's Hospital, Sydney' (4 aliases): 3 PMIDs found
INFO src.pubmed_search: Site enrichment 'Austin Health, Heidelberg' (1 aliases): 0 PMIDs found
INFO src.pubmed_search: Fetching PubMed records 1–57 / 57



Targeted enrichment: 57 new unique PMIDs (from 110 total retrieved)


INFO src.openalex_client: Enriching paper 1501/1557 (pmid=37088416)


...

Classification: 236 core_surgical, 982 surgery_adjacent
Rebuilding author co-authorship network...
Author network: 11151 authors
Re-matching affiliations (country filter removed; all countries considered)...


INFO src.author_network: Saved 11151 author rows to C:\Users\Richard Rosch\Documents\Code\neurosurgical-site-ranking\outputs\authors.csv
INFO src.site_matcher: Saved 32257 audit rows to C:\Users\Richard Rosch\Documents\Code\neurosurgical-site-ranking\outputs\match_audit.csv
INFO src.site_matcher: Saved 6013 unresolved rows to C:\Users\Richard Rosch\Documents\Code\neurosurgical-site-ranking\outputs\unresolved_affiliations.csv


Re-scoring candidate sites...


INFO src.site_scorer: Saved 37 site rows to C:\Users\Richard Rosch\Documents\Code\neurosurgical-site-ranking\outputs\site_scores.csv
INFO src.site_scorer: Saved 58 possible duplicate rows to C:\Users\Richard Rosch\Documents\Code\neurosurgical-site-ranking\outputs\possible_duplicate_authors.csv
INFO src.site_scorer: Saved 2 missing candidate rows to C:\Users\Richard Rosch\Documents\Code\neurosurgical-site-ranking\outputs\missing_candidate_sites.csv



After enrichment: 35 sites scored, 2 still unscored.
Still unscored:
           canonical_site country              evidence_status
   Mayo Clinic, Rochester     USA           alias_issue_likely
Austin Health, Heidelberg     AUS no_candidate_authors_matched


In [10]:
# ── 8b. Final scoring summary ─────────────────────────────────────────────────
import pandas as pd

print('=== Final site scores summary ===')
print(f'Total candidate sites: {len(site_scores)}')

by_status = site_scores.groupby('evidence_status')['canonical_site'].count()
for status, count in by_status.items():
    print(f'  {status}: {count}')

print('\nScored sites by country:')
scored = site_scores[site_scores['evidence_status'] == 'scored']
if not scored.empty:
    print(scored.groupby('country')[['canonical_site', 'composite_kol_score']]
          .apply(lambda g: g.set_index('canonical_site')['composite_kol_score'])
          .to_string())
else:
    print('  (none)')

print('\nTop 10 candidate sites by composite_kol_score:')
scored.head(10)[['canonical_site', 'country', 'composite_kol_score',
                 'n_kol_candidates', 'n_core_surgical_papers', 'evidence_status']]


=== Final site scores summary ===
Total candidate sites: 37
  alias_issue_likely: 1
  no_candidate_authors_matched: 1
  scored: 35

Scored sites by country:
country  canonical_site                                             
AUS      Royal Brisbane & Women's Hospital, Herston                     36.44
         Royal Melbourne Hospital / Melbourne Health, Melbourne         36.37
         St Vincent's Hospital, Fitzroy                                 21.86
         St Vincent's Hospital, Sydney                                  20.58
UK       King's College Hospital, London                                60.07
         Addenbrooke's Hospital, Cambridge                              46.77
         John Radcliffe Hospital, Oxford                                39.86
         North Bristol NHS Trust, Bristol                               34.27
         University College London Hospital (including NHNN), London    32.35
         Leeds Teaching Hospitals, Leeds                                

,canonical_site,country,composite_kol_score,n_kol_candidates,n_core_surgical_papers,evidence_status
0,"Johns Hopkins University, Baltimore",USA,93.97,89,12,scored
1,"Mayo Clinic, Jacksonville",USA,83.58,73,7,scored
2,"UCSF, San Francisco",USA,66.40,111,7,scored
3,"King's College Hospital, London",UK,60.07,77,6,scored
4,"University of Pittsburgh Medical Center, Pitts...",USA,59.01,61,8,scored
5,"Stanford, Stanford",USA,56.47,51,8,scored
6,"Cleveland Clinic, Cleveland",USA,56.32,42,2,scored
7,"Northwestern University, Chicago",USA,55.59,285,7,scored
8,"Duke University, Durham",USA,53.79,246,12,scored
9,"Emory University, Atlanta",USA,50.33,292,13,scored


In [11]:
# ── 9. KOL candidate list (top-30-centile site-affiliated authors) ─────────────
# Returns all site-affiliated authors whose author_kol_score is in the
# top 30 % of scores for that matched cohort (i.e. ≥ 70th percentile).

kol_candidates = scorer.kol_candidates(authors_df, top_centile=30.0)
scorer.save_kol_candidates(kol_candidates)  # outputs/kol_candidates.csv

print(f'{len(kol_candidates)} KOL candidates (top-30-centile, candidate-site-affiliated)')

# Show key columns for readability
display_cols = [
    c for c in [
        'kol_centile_rank', 'display_name', 'canonical_site',
        'author_kol_score', 'core_surgical_count', 'surgery_adjacent_count',
        'weighted_citation_score', 'site_confidence',
    ]
    if c in kol_candidates.columns
]
kol_candidates[display_cols]


INFO src.site_scorer: KOL candidates (top 30 %, candidate sites): 610 authors (score ≥ 4.69)
INFO src.site_scorer: Saved 610 KOL candidate rows to C:\Users\Richard Rosch\Documents\Code\neurosurgical-site-ranking\outputs\kol_candidates.csv


610 KOL candidates (top-30-centile, candidate-site-affiliated)


,kol_centile_rank,display_name,canonical_site,author_kol_score,core_surgical_count,surgery_adjacent_count,weighted_citation_score,site_confidence
0,1,Alfredo Quiñones‐Hinojosa,"Mayo Clinic, Jacksonville",56.01,3,8,92.27,1.000
1,2,Jordina Rincón-Torroella,"Johns Hopkins University, Baltimore",54.38,7,0,15.42,1.000
2,3,Chetan Bettegowda,"Johns Hopkins University, Baltimore",50.87,6,1,16.69,1.000
3,4,Peiwen Chen,"Cleveland Clinic, Cleveland",47.78,0,5,252.00,1.000
4,5,Shawn L. Hervey‐Jumper,"UCSF, San Francisco",47.47,4,6,36.02,1.000
...,...,...,...,...,...,...,...,...
605,606,Fan Xing,"Northwestern University, Chicago",4.76,0,1,7.00,0.724
606,607,Moustafa Mansour,"Emory University, Atlanta",4.76,0,1,7.00,0.744
607,608,Meng Jiao,"Duke University, Durham",4.74,0,1,8.00,1.000
608,609,Chuan‐Yuan Li,"Duke University, Durham",4.74,0,1,8.00,1.000


In [28]:
# # ── 10. Download results (Colab only) ─────────────────────────────────────────
# try:
#     from google.colab import files
#     for fname in ['outputs/papers.csv', 'outputs/authors.csv',
#                   'outputs/site_scores.csv', 'outputs/kol_candidates.csv',
#                   'outputs/match_audit.csv',
#                   'outputs/unresolved_affiliations.csv']:
#         try:
#             files.download(fname)
#         except Exception as e:
#             print(f'  Could not download {fname}: {e}')
# except ImportError:
#     print('Not in Colab – find output files in the outputs/ directory.')

In [12]:
# ── Smoke test ────────────────────────────────────────────────────────────────
# Validates real pipeline outputs.  Must FAIL (raise AssertionError) if:
#   • site_scores.csv contains non-candidate sites
#   • kol_candidates.csv contains non-candidate sites
#   • composite_kol_score is missing / non-numeric / negative / non-finite
#   • composite score cannot be reproduced from the 4 component columns
#   • targeted enrichment was enabled but no targeted search was run
#   • no UK or AUS sites scored after enrichment
import math
import os
import pandas as pd
from src.site_matcher import SiteMatcher
from src.site_scorer import SiteScorer

_aliases = ALIASES_CSV
_longlist = LONGLIST_CSV

# ── 1. Single deterministic affiliation match ─────────────────────────────
smoke_matcher = SiteMatcher(_aliases, use_llm=False)
test_aff = 'The University of Texas MD Anderson Cancer Center, Houston, TX'
smoke_result = smoke_matcher.match_affiliation(test_aff, author_id='smoke_test')
assert smoke_result['canonical_site'] != '', \
    f'SMOKE FAIL: expected a site match for "{test_aff}"'
print(f'1. Match: {smoke_result["canonical_site"]} '
      f'(conf={smoke_result["confidence"]:.3f}, method={smoke_result["match_method"]})')

# ── 2. Scorer loads candidate sites from longlist ─────────────────────────
smoke_scorer = SiteScorer(longlist_csv=_longlist, aliases_csv=_aliases)
assert len(smoke_scorer.candidate_sites) > 0, \
    'SMOKE FAIL: no candidate sites loaded from longlist'
print(f'2. Candidate sites loaded: {len(smoke_scorer.candidate_sites)}')

# ── 3. Non-candidate sites excluded from outputs ──────────────────────────
smoke_df = pd.DataFrame([
    {
        'author_id': 'A1', 'display_name': 'Alice Smith',
        'author_kol_score': 80.0, 'core_surgical_count': 5,
        'surgery_adjacent_count': 2, 'pagerank': 0.01,
        'degree_centrality': 0.05, 'centrifugal_score': 0.8,
        'recency_score': 0.9, 'weighted_citation_score': 200.0,
        'canonical_site': smoke_result['canonical_site'],
        'site_confidence': 0.95, 'match_method': 'exact_alias',
    },
    {
        'author_id': 'A2', 'display_name': 'Bob Jones',
        'author_kol_score': 60.0, 'core_surgical_count': 2,
        'surgery_adjacent_count': 1, 'pagerank': 0.005,
        'degree_centrality': 0.02, 'centrifugal_score': 0.5,
        'recency_score': 0.7, 'weighted_citation_score': 80.0,
        'canonical_site': 'Random Non-Candidate Institute',
        'site_confidence': 0.90, 'match_method': 'fuzzy',
    },
])
smoke_scores = smoke_scorer.compute(smoke_df)
assert 'Random Non-Candidate Institute' not in smoke_scores['canonical_site'].values, \
    'SMOKE FAIL: non-candidate site leaked into site_scores'
print('3. Non-candidate site correctly excluded from site_scores.')

# ── 4. Missing-candidate audit returns correct DataFrame ──────────────────
smoke_missing = smoke_scorer.missing_candidates(smoke_scores)
assert isinstance(smoke_missing, pd.DataFrame) and 'canonical_site' in smoke_missing.columns, \
    'SMOKE FAIL: missing_candidates() did not return expected DataFrame'
print(f'4. Missing candidate audit: {len(smoke_missing)} unscored sites.')

# ── 5. KOL candidates contain only candidate-site authors ─────────────────
smoke_kol = smoke_scorer.kol_candidates(smoke_df, top_centile=50.0)
if not smoke_kol.empty:
    non_cand = smoke_kol[~smoke_kol['canonical_site'].isin(smoke_scorer.candidate_sites)]
    assert non_cand.empty, \
        f'SMOKE FAIL: non-candidate sites in kol_candidates: {non_cand["canonical_site"].tolist()}'
print(f'5. KOL candidates: {len(smoke_kol)} (candidate-site-only).')

# ── 6. Validate real output files ─────────────────────────────────────────
output_files = {
    'outputs/site_scores.csv':              True,
    'outputs/kol_candidates.csv':           True,
    'outputs/missing_candidate_sites.csv':  True,
    'outputs/possible_duplicate_authors.csv': True,
}
for f, required in output_files.items():
    if os.path.exists(f):
        print(f'  ✓ {f} ({os.path.getsize(f)} bytes)')
    elif required:
        raise AssertionError(f'SMOKE FAIL: required output file missing: {f}')
    else:
        print(f'  – {f} not found')

# ── 7. Validate site_scores.csv schema and composite formula ─────────────
ss = pd.read_csv('outputs/site_scores.csv')

# Required columns
required_cols = [
    'canonical_site', 'country', 'region', 'composite_kol_score',
    'site_neurosurgical_kol_strength', 'site_network_engagement',
    'site_depth', 'site_confidence',
    'n_kol_candidates', 'n_core_surgical_papers', 'n_surgery_adjacent_papers',
    'top_kol_names', 'evidence_status',
]
missing_cols = [c for c in required_cols if c not in ss.columns]
assert not missing_cols, f'SMOKE FAIL: site_scores.csv missing columns: {missing_cols}'
print(f'6. site_scores.csv has all required columns ({len(ss)} rows).')

# All rows must be candidate sites
longlist_df = pd.read_csv(_longlist)
site_col = next((c for c in longlist_df.columns if c.lower() == 'site'), longlist_df.columns[0])
candidate_set = set(longlist_df[site_col].dropna().str.strip())
non_cand_rows = ss[~ss['canonical_site'].isin(candidate_set)]
assert non_cand_rows.empty, \
    f'SMOKE FAIL: non-candidate sites in site_scores.csv: {non_cand_rows["canonical_site"].tolist()}'
print(f'7. All {len(ss)} rows in site_scores.csv are candidate sites.')

# For scored sites: composite_kol_score must be numeric, non-negative, finite
scored_ss = ss[ss['evidence_status'] == 'scored'].copy()
assert not scored_ss.empty, 'SMOKE FAIL: no scored sites in site_scores.csv'
bad_scores = scored_ss[
    scored_ss['composite_kol_score'].isna()
    | (scored_ss['composite_kol_score'] < 0)
    | ~scored_ss['composite_kol_score'].apply(math.isfinite)
]
assert bad_scores.empty, \
    f'SMOKE FAIL: invalid composite_kol_score values:\n{bad_scores[["canonical_site","composite_kol_score"]]}'
print(f'8. composite_kol_score: numeric and finite for all {len(scored_ss)} scored sites.')

# Composite score must be reproducible from 4 components
_W = {'site_neurosurgical_kol_strength': 0.45,
      'site_network_engagement': 0.25,
      'site_depth': 0.20,
      'site_confidence': 0.10}
tol = 0.15   # rounding tolerance
for _, row in scored_ss.iterrows():
    expected = sum(w * row[col] for col, w in _W.items()) * 100
    actual = row['composite_kol_score']
    if not math.isnan(expected) and abs(expected - actual) > tol:
        raise AssertionError(
            f'SMOKE FAIL: composite score mismatch for {row["canonical_site"]}: '
            f'expected {expected:.2f}, got {actual:.2f}'
        )
print('9. Composite score reproducible from 4 components for all scored sites.')

# ── 8. Validate kol_candidates.csv ────────────────────────────────────────
kc = pd.read_csv('outputs/kol_candidates.csv')
non_cand_kol = kc[~kc['canonical_site'].isin(candidate_set)]
assert non_cand_kol.empty, \
    f'SMOKE FAIL: non-candidate sites in kol_candidates.csv: {non_cand_kol["canonical_site"].tolist()}'
print(f'10. kol_candidates.csv: {len(kc)} rows, all candidate-site authors.')

# ── 9. Targeted enrichment audit ─────────────────────────────────────────
if RUN_MISSING_SITE_ENRICHMENT:
    assert _enrichment_ran, \
        'SMOKE FAIL: RUN_MISSING_SITE_ENRICHMENT=True but no targeted search was run.'
    print('11. Targeted enrichment ran as required.')

# ── 10. UK / AUS coverage check ───────────────────────────────────────────
UK_AUS_COUNTRIES = {'UK', 'Australia'}
uk_aus_scored = ss[
    ss['country'].isin(UK_AUS_COUNTRIES)
    & (ss['evidence_status'] == 'scored')
]
if uk_aus_scored.empty:
    msg = ('SMOKE TEST WARNING: zero UK/AUS candidate sites scored after enrichment; '
           'likely alias/search failure.')
    if RUN_MISSING_SITE_ENRICHMENT:
        raise AssertionError(f'SMOKE FAIL: {msg}')
    else:
        print(msg)
else:
    print(f'12. UK/AUS coverage: {len(uk_aus_scored)} site(s) scored.')
    print(uk_aus_scored[['canonical_site', 'country', 'composite_kol_score']].to_string(index=False))

print('\nSMOKE TEST PASSED')


INFO src.site_matcher: Loaded 123 canonical sites, 564 aliases
INFO src.site_scorer: Loaded 37 candidate sites from c:\Users\Richard Rosch\Documents\Code\neurosurgical-site-ranking\data\site_longlist.csv
INFO src.site_scorer: KOL candidates (top 50 %, candidate sites): 1 authors (score ≥ 80.00)


1. Match: MD Anderson Cancer Center, Houston (conf=0.950, method=exact_alias)
2. Candidate sites loaded: 37
3. Non-candidate site correctly excluded from site_scores.
4. Missing candidate audit: 36 unscored sites.
5. KOL candidates: 1 (candidate-site-only).
  ✓ outputs/site_scores.csv (13804 bytes)
  ✓ outputs/kol_candidates.csv (115606 bytes)
  ✓ outputs/missing_candidate_sites.csv (177 bytes)
  ✓ outputs/possible_duplicate_authors.csv (6118 bytes)
6. site_scores.csv has all required columns (37 rows).
7. All 37 rows in site_scores.csv are candidate sites.
8. composite_kol_score: numeric and finite for all 35 scored sites.
9. Composite score reproducible from 4 components for all scored sites.
10. kol_candidates.csv: 610 rows, all candidate-site authors.
11. Targeted enrichment ran as required.
12. UK/AUS coverage: 8 site(s) scored.
                                             canonical_site country  composite_kol_score
                            King's College Hospital, London      